<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/rnn/wip-rnn-bit-parity-classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN - Bit-parity classifier

# Setup

In [69]:
!pip install wandb tsilva-notebook-utils==0.0.15 > /dev/null

Loading API keys and authentication tokens from Colab secrets:


In [70]:
from tsilva_notebook_utils.colab import load_secrets_into_env

load_secrets_into_env([
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

This notebook demonstrates building an RNN to classify bit sequences based on parity. The model will:
- Output `1` for sequences with an odd number of `1`s 
- Output `0` for sequences with an even number of `1`s


In [71]:
import os
from tsilva_notebook_utils.colab import notebook_id_from_title

def setup_config():
    #@markdown Random seed for reproducibility
    seed = 42  # @param {type:"integer"}

    #@markdown Number of training epochs
    n_epochs = 10000 # @param {type:"integer"}

    #@markdown Batch size
    batch_size = 64  # @param {type:"integer"}

    #@markdown Learning rate for the optimizer
    learning_rate = 0.0005  # @param {type:"number"}

    #@markdown Length of the input sequence (number of time steps)
    sequence_length = 10  # @param {type:"integer"}

    #@markdown Number of hidden units in the RNN
    hidden_size = 64  # @param {type:"integer"}

    #@markdown Activation function to use in the RNN ('tanh' or 'relu')
    nonlinearity = 'tanh'  # @param ['tanh', 'sigmoid', 'relu']

    #@markdown Weight initialization strategy ('none', 'xavier', or 'kaiming')
    weight_init = 'none'  # @param ['none', 'xavier', 'kaiming']

    #@markdown Maximum gradient norm for clipping (0.0 means no clipping)
    max_grad_norm = 0  # @param {type:"number"}

    # Calculate train/val split sizes (e.g., 80% train, 20% val)
    train_size = 0.8
    val_size = 0.2

    # These are meant to be hardcoded in this notebook
    batch_size = 32
    input_size = 1 # One input, one digit at a time is passed through RNN
    output_size = 2 # Two outputs: one for even, another for odd

    # Generate notebook id from notebook title
    os.environ["NOTEBOOK_ID"] = notebook_id_from_title()

    return {
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'sequence_length': sequence_length,
        'input_size': input_size,
        'hidden_size': hidden_size,
        'output_size': output_size,
        'nonlinearity': nonlinearity,
        'weight_init': weight_init,
        'max_grad_norm': max_grad_norm,
        'train_size': train_size,
        'val_size': val_size
    }

CONFIG = setup_config()

Setting seed for reproducible results:


In [72]:
from tsilva_notebook_utils.colab import set_seed
set_seed(CONFIG['seed'])

Generating parity dataset with binary sequences:


In [73]:
import random
import torch
import itertools
from sklearn.model_selection import train_test_split
import numpy as np
from torch.utils.data import TensorDataset
from tsilva_notebook_utils.torch import inspect_tensor_dataset

def generate_parity_dataset(
    sequence_length=None,
    val_size=None,
    seed=None
):
    if sequence_length is None: sequence_length = CONFIG['sequence_length']
    if val_size is None: val_size = CONFIG['val_size']
    if seed is None: seed = CONFIG['seed']

    # Set seed for reproducibility
    set_seed(seed)

    # Generate all unique binary sequences
    all_sequences = list(itertools.product([0, 1], repeat=sequence_length))
    data = []
    labels = []

    for seq in all_sequences:
        parity = sum(seq) % 2
        data.append(seq)
        labels.append(parity)

    X = torch.tensor(data, dtype=torch.float32).unsqueeze(-1)  # shape: (N, sequence_length, 1)
    Y = torch.tensor(labels, dtype=torch.long)

    # Shuffle the data
    perm = torch.randperm(len(X))
    X = X[perm]
    Y = Y[perm]

    # Split the dataset
    X_train, X_val, Y_train, Y_val = train_test_split(
        X, Y, test_size=val_size, random_state=seed, shuffle=True
    )

    train_dataset = TensorDataset(X_train, Y_train)
    val_dataset = TensorDataset(X_val, Y_val)

    return train_dataset, val_dataset

train_dataset, val_dataset = generate_parity_dataset()
inspect_tensor_dataset(train_dataset)

{'num_samples': 819,
 'tensor_shapes': [torch.Size([819, 10, 1]), torch.Size([819])],
 'sample_data': [(tensor([[1.],
           [0.],
           [0.],
           [0.],
           [1.],
           [0.],
           [1.],
           [0.],
           [1.],
           [0.]]),
   tensor(0)),
  (tensor([[0.],
           [1.],
           [0.],
           [0.],
           [1.],
           [1.],
           [1.],
           [0.],
           [1.],
           [0.]]),
   tensor(1)),
  (tensor([[1.],
           [1.],
           [0.],
           [1.],
           [1.],
           [1.],
           [1.],
           [0.],
           [0.],
           [0.]]),
   tensor(0))]}

Creating DataLoaders for efficient batch processing:


In [74]:
from torch.utils.data import TensorDataset, DataLoader

# Create DataLoader
batch_size = CONFIG['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Sample a batch (iterator)
X_batch, Y_batch = next(iter(train_loader))
X_batch.shape, Y_batch.shape

(torch.Size([32, 10, 1]), torch.Size([32]))

Building the RNN model with customizable architecture:


In [75]:
import torch
import torch.nn as nn

class ParityRNN(nn.Module):
    def __init__(
        self,
        input_size=None,
        hidden_size=None,
        output_size=None,
        nonlinearity=None,
        weight_init=None
    ):
        super(ParityRNN, self).__init__()

        if input_size is None: input_size = CONFIG['input_size']
        if hidden_size is None: hidden_size = CONFIG['hidden_size']
        if output_size is None: output_size = CONFIG['output_size']
        if nonlinearity is None: nonlinearity = CONFIG['nonlinearity']
        if weight_init is None: weight_init = CONFIG['weight_init']

        # RNN layer (no bias since you're using manual bias before)
        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            batch_first=True,
            nonlinearity=nonlinearity
        )

        # Output projection layer
        self.fc = nn.Linear(hidden_size, output_size)

        # Apply weight initialization
        self.init_weights(weight_init, nonlinearity)

    def init_weights(self, weight_init, nonlinearity):
        if weight_init == 'none':
            return

        for name, param in self.rnn.named_parameters():
            if 'weight' in name:
                if weight_init == 'xavier':
                    nn.init.xavier_uniform_(param, gain=nn.init.calculate_gain(nonlinearity))
                elif weight_init == 'kaiming':
                    nn.init.kaiming_uniform_(param, mode='fan_in', nonlinearity=nonlinearity)
            elif 'bias' in name:
                nn.init.zeros_(param)

        if weight_init == 'xavier':
            nn.init.xavier_uniform_(self.fc.weight, gain=nn.init.calculate_gain(nonlinearity))
        elif weight_init == 'kaiming':
            nn.init.kaiming_uniform_(self.fc.weight, mode='fan_in', nonlinearity=nonlinearity)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        hidden_states, last_hidden_state = self.rnn(x)
        logits = self.fc(last_hidden_state.squeeze(0))
        return logits, hidden_states

model = ParityRNN()
logits, _ = model(X_batch)
logits

tensor([[-0.1411,  0.0372],
        [-0.1262,  0.0364],
        [-0.1422,  0.0154],
        [-0.1306,  0.0365],
        [-0.1395,  0.0274],
        [-0.1432,  0.0512],
        [-0.1471,  0.0505],
        [-0.1467,  0.0277],
        [-0.1579,  0.0293],
        [-0.1457,  0.0121],
        [-0.1531,  0.0541],
        [-0.1402,  0.0283],
        [-0.1580,  0.0506],
        [-0.1400,  0.0133],
        [-0.1378,  0.0148],
        [-0.1230,  0.0146],
        [-0.1310,  0.0376],
        [-0.1334,  0.0118],
        [-0.1425,  0.0384],
        [-0.1415,  0.0301],
        [-0.1482,  0.0139],
        [-0.1300,  0.0112],
        [-0.1491,  0.0522],
        [-0.1305,  0.0361],
        [-0.1645,  0.0284],
        [-0.1394,  0.0382],
        [-0.1477,  0.0277],
        [-0.1363,  0.0530],
        [-0.1297,  0.0341],
        [-0.1603,  0.0267],
        [-0.1242,  0.0361],
        [-0.1394,  0.0141]], grad_fn=<AddmmBackward0>)

Initializing Weights & Biases (wandb) for experiment tracking:


In [76]:
from tsilva_notebook_utils.wandb import init_with_defaults
init_with_defaults(CONFIG)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Training the model with early stopping when perfect accuracy is achieved:


In [77]:
import torch
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm
import wandb

def train(
    n_epochs=None,
    learning_rate=None,
    max_grad_norm=None
):
    if n_epochs is None: n_epochs = CONFIG['n_epochs']
    if learning_rate is None: learning_rate = CONFIG['learning_rate']
    if max_grad_norm is None: max_grad_norm = CONFIG['max_grad_norm']

    # Set model in training mode
    model.train()

    # Optionally: Watch the model to log gradients and model topology
    wandb.watch(model, log="all")

    # Define the loss function as Mean Squared Error loss
    loss_fn = nn.CrossEntropyLoss()

    # Initialize the Adam optimizer with model parameters and the learning rate
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Training loop with progress bar using tqdm
    with tqdm(range(n_epochs), desc="Training") as pbar:
        for epoch in pbar:
            losses = []
            accuracies = []

            # Iterate over batches of data from the training data loader
            for x_batch, y_batch in train_loader:
                # Forward pass: compute model predictions and capture hidden states
                logits, hidden_states = model(x_batch)

                # Calculate the loss
                loss = loss_fn(logits, y_batch)
                losses.append(loss.item())

                # Backpropagation step
                optimizer.zero_grad()  # Clear previous gradients
                loss.backward()        # Compute gradients

                # Apply gradient clipping if enabled (max_grad_norm > 0)
                if max_grad_norm > 0:
                    clip_grad_norm_(model.parameters(), max_grad_norm)

                optimizer.step()       # Update model parameters

                # Calculate the accuracy
                with torch.no_grad():
                    predicted = torch.argmax(logits, dim=1)
                    accuracy = (predicted == y_batch).float().mean().item()
                    accuracies.append(accuracy)

            # Compute average stats for this epoch
            avg_loss = sum(losses) / len(train_loader)
            avg_accuracy = sum(accuracies) / len(accuracies)

            # Log average loss to wandb
            wandb.log({'epoch': epoch + 1, 'train_loss': avg_loss, 'train_accuracy' : avg_accuracy})

            # Update the progress bar with the current epoch and average loss
            pbar.set_postfix({'epoch': epoch + 1, 'train_loss': f'{avg_loss:.6f}', 'train_accuracy': f'{avg_accuracy * 100:.2f}%'})

            if avg_accuracy == 1.0:
                print("\nTraining finished, model memorized dataset.")
                break

    wandb.finish()

train()

Training:  18%|█▊        | 1842/10000 [03:27<15:17,  8.89it/s, epoch=1843, loss=0.025142, accuracy=100.00%]


Training finished, model memorized dataset.


accuracy,▁▁▁▁▁▁▁▁▁▁▁▂▂▁▂▂▂▂▂▂▂▂▂▂▂▄▄▅▅▆▆▇▇▇▇█████
epoch,▁▁▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▆▆▇▇▇██
loss,██████████████████████████▇▇▇▇▆▆▆▅▅▃▂▁▁▁
accuracy,1
epoch,1843
loss,0.02514


Evaluating model with manual input testing:


In [ ]:
def manual_test():
    while True:
        sequence_s = input()
        if sequence_s == "exit": return
        x = torch.tensor(list(map(int, sequence_s))).unsqueeze(-1).float()
        with torch.no_grad(): logits, _ = model(x)
        prediction = torch.argmax(logits, dim=0)
        prediction = prediction.item()
        prediction_s = "even" if prediction == 0 else "odd"
        print(prediction_s)

# Uncomment this line to manually test predictions
# (commented by default to allow running `notify_and_disconnect_after_timeout` in next cell)
manual_test()

1
even
0
even
100000000
odd
1100000000
even
1000000001
even
1010000000
even


Setting up auto-notification and resource management:


In [ ]:
from tsilva_notebook_utils.colab import notify_and_disconnect_after_timeout
notify_and_disconnect_after_timeout()